In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
df = sns.load_dataset("flights")
df.to_csv(r"flights.csv", index = False)

In [ ]:
print(df.head())

In [ ]:
df["month"].value_counts()

In [ ]:
months = {
    "Jan": "January",
    "Feb": "February",
    "Mar": "March",
    "Apr": "April",
    "May": "May",
    "Jun": "June",
    "Jul": "July",
    "Aug": "August",
    "Sep": "September",
    "Oct": "October",
    "Nov": "November",
    "Dec": "December"
}

df["month"] = df["month"].map(months)

In [ ]:
df["month"].unique()

In [ ]:
def seasons(month):
    if month in ["June", "July", "August"]:
        return "Summer"
    elif month in ["September", "October", "November"]:
        return "Fall"
    elif month in ["December", "January", "February"]:
        return "Winter"
    else:
        return "Spring"

df["season_apply"] = df["month"].apply(seasons)

In [ ]:
conditions = [
    df["month"].isin(["June", "July", "August"]),
    df["month"].isin(["September", "October", "November"]),
    df["month"].isin(["December", "January", "February"]),
    df["month"].isin(["March", "April", "May"])
]

choices = ["Summer", "Fall", "Winter", "Spring"]

df["season_vectorized"] = np.select(conditions, choices, default="Spring").astype(str)

In [ ]:
df["season_vectorized"] = df["season_vectorized"].str.lower()

In [ ]:
df["peak_season"] = np.where(df["season_apply"] == "Summer", 1, 0)

In [ ]:
df["season"] = df["season_apply"].str.lower()

In [ ]:
df["date"] = pd.to_datetime(df["year"].astype(str) + "-" + df["month"].astype(str), format="%Y-%B")
df = df.sort_values("date")
df["growth"] = df["passengers"].pct_change()

In [ ]:
df_big = pd.concat([df] * 1000, ignore_index=True)

In [ ]:
import time

start = time.time()
df_big["season_apply"] = df_big["month"].apply(seasons)
t_apply = time.time() - start

start = time.time()
conditions = [
    df_big["month"].isin(["June", "July", "August"]),
    df_big["month"].isin(["September", "October", "November"]),
    df_big["month"].isin(["December", "January", "February"]),
    df_big["month"].isin(["March", "April", "May"])
]
choices = ["Summer", "Fall", "Winter", "Spring"]
df_big["season_vectorized"] = np.select(conditions, choices, default="Spring").astype(str)
df_big["season_vectorized"] = df_big["season_vectorized"].str.lower()
t_vector = time.time() - start

print("Apply time:", t_apply)
print("Vectorized time:", t_vector)

#Apply time: 0.008993864059448242
#Vectorized time: 0.08195281028747559

In [ ]:
print(df.tail())

In [ ]:
df= df.drop(columns = ["season_apply", "season_vectorized", "date"])

In [ ]:
print(df.head())

In [ ]:
df.to_csv(r"flights_final.csv", index = False)